In [ ]:
import openai
from azure.storage.blob import BlobServiceClient
import pickle
import numpy as np
import os

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchFieldDataType, SearchableField
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
import re

In [ ]:
query_documents = [
    {
        "id": "query_001",
        "query_desc": "Get all employees with their department names",
        "sql_query": """
            SELECT e.emp_id, e.emp_name, e.emp_salary, d.dept_name
            FROM employee e
            LEFT JOIN department d ON e.dept_id = d.dept_id;
        """
    },
    {
        "id": "query_002",
        "query_desc": "Find the highest-paid employee",
        "sql_query": """
            SELECT emp_name, emp_salary 
            FROM employee 
            ORDER BY emp_salary DESC 
            LIMIT 1;
        """
    },
    {
        "id": "query_003",
        "query_desc": "Count the number of employees in each department",
        "sql_query": """
            SELECT d.dept_name, COUNT(e.emp_id) as employee_count
            FROM department d
            LEFT JOIN employee e ON d.dept_id = e.dept_id
            GROUP BY d.dept_name;
        """
    },
    {
        "id": "query_004",
        "query_desc": "Get total salary expense for each department",
        "sql_query": """
            SELECT d.dept_name, SUM(e.emp_salary) as total_salary
            FROM department d
            LEFT JOIN employee e ON d.dept_id = e.dept_id
            GROUP BY d.dept_name;
        """
    },
    {
        "id": "query_005",
        "query_desc": "Find employees who earn more than 70,000",
        "sql_query": """
            SELECT emp_id, emp_name, emp_salary
            FROM employee
            WHERE emp_salary > 70000;
        """
    },
    {
        "id": "query_006",
        "query_desc": "Get employees who do not belong to any department",
        "sql_query": """
            SELECT emp_id, emp_name
            FROM employee
            WHERE dept_id IS NULL;
        """
    },
    {
        "id": "query_007",
        "query_desc": "Find the average salary per department",
        "sql_query": """
            SELECT d.dept_name, AVG(e.emp_salary) as avg_salary
            FROM department d
            LEFT JOIN employee e ON d.dept_id = e.dept_id
            GROUP BY d.dept_name;
        """
    },
    {
        "id": "query_008",
        "query_desc": "Retrieve employees whose name starts with 'J'",
        "sql_query": """
            SELECT emp_id, emp_name
            FROM employee
            WHERE emp_name LIKE 'J%';
        """
    }
]


In [ ]:
# Azure Cognitive Search details
search_service_name = ""
admin_api_key = ""
query_index_name = "query_index" #define new name

# Initialize Search Index client
endpoint = f"https://{search_service_name}.search.windows.net"
credential = AzureKeyCredential(admin_api_key)
index_client = SearchIndexClient(endpoint=endpoint, credential=credential)

# Define the index schema
def create_search_index():
    try:
        fields = [
            SimpleField(name="id", type=SearchFieldDataType.String, key=True),
            SearchableField(name="query_desc", type=SearchFieldDataType.String),
            SimpleField(name="sql_query", type=SearchFieldDataType.String)
        ]
        
        index = SearchIndex(name=query_index_name, fields=fields)
        index_client.create_index(index)
        print("✅ Query Index Created!")
    except Exception as e:
        print(f" Error Query Index Created!: {e}")
        raise

create_search_index()


In [ ]:
query_client = SearchClient(
            endpoint=endpoint,
            index_name=query_index_name,
            credential=credential
        )

# Upload Updated Queries

query_client.upload_documents(query_documents)
print("✅ Updated Queries Uploaded!")


In [ ]:
# 🔹 Schema Data
schema_documents = [
    {
        "id": "schema_employee",
        "table_name": "employee",
        "schema_metadata": """
            Table: employee
            Columns: emp_id (INT, PK), emp_name (VARCHAR), emp_salary (DECIMAL), dept_id (INT, FK to department)
        """
    },
    {
        "id": "schema_department",
        "table_name": "department",
        "schema_metadata": """
            Table: department
            Columns: dept_id (INT, PK), dept_name (VARCHAR)
        """
    }
]

In [ ]:
# Azure Cognitive Search details
search_service_name = ""
admin_api_key = ""
schema_index_name = "schema_index" #define new name

# Initialize Search Index client
endpoint = f"https://{search_service_name}.search.windows.net"
credential = AzureKeyCredential(admin_api_key)
index_client = SearchIndexClient(endpoint=endpoint, credential=credential)

# Define the index schema
def create_search_index():
    try:
        fields = [
           SimpleField(name="id", type=SearchFieldDataType.String, key=True),
            SearchableField(name="table_name", type=SearchFieldDataType.String,  analyzer_name="standard.lucene"),
            SearchableField(name="schema_metadata", type=SearchFieldDataType.String,  analyzer_name="standard.lucene")
        ]
        
        index = SearchIndex(name=schema_index_name, fields=fields)
        index_client.create_index(index)
        print("✅ Query Index Created!")
    except Exception as e:
        print(f" Error Query Index Created!: {e}")
        raise

create_search_index()


In [ ]:
schema_client = SearchClient(
            endpoint=endpoint,
            index_name=schema_index_name,
            credential=credential
        )

# Upload Updated Queries

schema_client.upload_documents(schema_documents)
print("✅ Updated Queries Uploaded!")


In [ ]:
index_name = "query_index"
def query_search(query_text, top_k=3):
    try:
        search_client = SearchClient(
            endpoint=endpoint,
            index_name=index_name,
            credential=credential
        )
        
        results = search_client.search(query_text, top=top_k)
        return  results #[(result["content"], result, result["@search.score"]) for result in results]
    except Exception as e:
        print(f"Error performing search: {e}")
        raise

# Example usage
query_text = "get all employees"
results = query_search(query_text)

for result in results:
    #text, metadata, score = result
    print(f"id: {result["id"]}")
    print(f"query_desc: {result["query_desc"]}")
    print(f"sql_query: {result["sql_query"]}")
    print(f"Score: {result["@search.score"]}\n")


In [ ]:
index_name = "schema_index"

def schema_search(query_text, top_k=2):
    try:
        search_client = SearchClient(
            endpoint=endpoint,
            index_name=index_name,
            credential=credential
        )
        
        results = search_client.search(query_text, search_mode="any", query_type="full", top=top_k)
        return results #[(result["content"], result, result["@search.score"]) for result in results]
    except Exception as e:
        print(f"Error performing search: {e}")
        raise

# Example usage
query_text = "get all employees"
query_text = "employee"
wildcard_query = f"{query_text}~"
results = schema_search(wildcard_query)

for result in results:
    #text, metadata, score = result
    print(f"id: {result["id"]}")
    print(f"table_name: {result["table_name"]}")
    print(f"schema_metadata: {result["schema_metadata"]}")
    print(f"Score: {result["@search.score"]}\n")
